El siguiente ejercicio carga un set de datos inmoviliarios donde se consideraron:
metros cuadrados de terreno, total de metros cuadrados de las habitaciones, total de metros cuadrados de los banos,
total de metros cuadrados de areas comunes (cocina, sala, comedor, etc) y el valor de la residencia en miles de dólares americanos. Sin embargo, solo se utilizará una sola variable (metros cuadrados de terreno) para determinar el precio de la casa. 

El objetivo general es hacer un modelo de regresión lineal unidimensional para determinar el precio de las residencias.
Los objetivos específicos son:
- Aprender a ejecutar un modelo de regresión linear multiparamétrico.
- Hacer un single split de los datos en relación 70-30 (entrenamiento-prueba)
- Entrenar el modelo.
- Hacer las predicciones con el modelo.
- Calcular las métricas y analizarlas.
- Mostrar porcentaje de aciertos con los datos de prueba.
- Realizar un gráfico de dispersión de los datos y su ajuste.

Nota 0: Un modelo de regresión lineal unidimensional se refiere a aquel modelo que se comporta como una línea recta y = mx + b; donde existe una sola variable dependiente y una sola variable independiente. El precio de la casa es la variable dependiente y el tamaño del terreno en metros cuadrados es la variable independiente.

Nota 1: Los datos ya fueron filtrados y arreglados.

Nota 2: El archivo .toml tiene toda la paquetería utilizada y que se instaló en automático en este proyecto.

## 1. Importaciones básicas

In [22]:
###### Importaciones básicas ###########
import pandas as pd
import matplotlib.pyplot as plt
import tkinter as tk
################### Importar Sklearn #####################
from sklearn.model_selection import train_test_split  # Se usa para hacer el single split (dividir los datos en entrenamiento y prueba)
from sklearn.linear_model import LinearRegression # Modelo de regresión linear
from sklearn.metrics import (r2_score,
                             mean_squared_error,
                             mean_absolute_error,
                             median_absolute_error,
                             mean_absolute_percentage_error) # Todas las métricas de evaluación para regresión linear

## 2. Cargar dataset

In [2]:
data_route = "../data/house_prices.csv" # Ruta del dataset
data = pd.read_csv(data_route) # Cargar el dataset

## 3. Explorar los datos

In [3]:
# Mostrar las primeras filas del dataset
data.head() # Mostrar las primeras filas del dataset

,terreno_m2,habitaciones_m2,banos_m2,areas_comunes_m2,precio_usd
0,85,40,12,33,210
1,120,55,18,47,320
2,95,38,10,30,180
3,110,50,15,40,270
4,130,60,20,50,350


In [4]:
# Mostrar info básica del dataset
data.info() # Mostrar info básica del dataset (solo para confirmar que no hay datos nulos o algún problema)

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   terreno_m2        100 non-null    int64
 1   habitaciones_m2   100 non-null    int64
 2   banos_m2          100 non-null    int64
 3   areas_comunes_m2  100 non-null    int64
 4   precio_usd        100 non-null    int64
dtypes: int64(5)
memory usage: 4.0 KB


Nota: Nunca se debe confiar en los datos; aunque nos digan que los datos están limpios, siempre hay que revisar el dataset para asegurarnos 
de que no hay datos nulos o algún problema que pueda afectar el modelo.

De la exploración simple de los datos se observó que hay un rango diferente de valores entre las características (X) y la variable objetivo (Y), lo que puede afectar el rendimiento del modelo. En este caso, se podría considerar estandarizar las características para mejorar el rendimiento del modelo porque el área del terreno será considerablemente mayor que al área de las habitaciones, baños y áreas comunes. Por otra parte, no es necesario estandarizar la variable objetivo (precio) porque está dentro del mismo orden de magnitud que las características (me refiero a que se trata de cientos; por otra parte si la variable objetivo estuviera en rangos de 100000; entonces si valdría la pena estandarizar).

NOTA IMPORTANTE: Los datos se deben escalar después de hacer el split para evitar el "Data Leakeage"
Por otra parte, como no se sabe que preprocesamiento necesitarán los datos (Normalizar, Estandarizar, Escalar, Binarizar); recién que exploré los datos; es que decido hacer la importación del módulo necesario

## 4. Realizar el preprocesado

In [5]:
from sklearn.preprocessing import StandardScaler # Importar el StandardScaler para normalizar los datos

In [6]:
# Separar las características (X) de la variable objetivo (Y)
X = data.iloc[:, :-1] # Características (todas las columnas excepto "precio_usd")
Y = data["precio_usd"] # Variable objetivo (columna "precio_usd en miles de dólares")

In [7]:
# Siempre es bueno revisar que lo que hacemos está correcto
print(X.head(2))

   terreno_m2  habitaciones_m2  banos_m2  areas_comunes_m2
0          85               40        12                33
1         120               55        18                47


In [8]:
# Siempre es bueno revisar que lo que hacemos está correcto
print(Y.head(2))

0    210
1    320
Name: precio_usd, dtype: int64


## 5. Crear el split y estandarizar los datos

In [9]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=1) # Hacer el single split (dividir los datos en entrenamiento y prueba)
scaler = StandardScaler() # Crear el scaler;(ajustado automáticamente a media = 0 y desviación estandar = 1)
scaler.fit(X_train) # Ajustar el scaler a los datos de entrenamiento (calcular la media y desviación estandar de cada característica en los datos de entrenamiento)
X_train_scaled = scaler.transform(X_train) # Transformar los datos de entrenamiento usando el scaler ajustado
X_test_scaled = scaler.transform(X_test) # Transformar los datos de prueba usando el scaler ajustado

## 6. Crear el modelo de regresión linear y ajustarlo a los datos de entrenamiento

In [10]:
model = LinearRegression() # Crear el modelo de regresión linear [Nota: Esto es una instancia del modelo]
model.fit(X_train_scaled, Y_train) # Entrenar el modelo elegido con los datos de entrenamiento (ya estandarizados)
Y_pred = model.predict(X_test_scaled) # Hacer predicciones con el modelo entrenado usando los datos de prueba (ya estandarizados)

## 7. Graficar los resultados (predicciones vs valores reales)
Opcional; en lo particular yo odio que los gráficos se muestren dentro del notebook, así que prefiero usar el siguiente comando para que se muestren en una ventana aparte. 

Para que esto funcione se debe tener instalado el paquete "python3-tk" (en Linux) o "tkinter" (en Windows).

Para instalar en linux: escribir en la terminal sudo apt-get install python3-tk

Para instalar en Windows; por lo general ya viene instalado por defecto al instalar python, pero si no lo tienes, puedes instalarlo con el comando pip install tk

Si instalaste tkinter ahora debes reiniciar el kernel para que funcione

Si quieres que la figura se muestren dentro del notebook, puedes comentar la siguiente línea.

In [11]:
%matplotlib tk

In [12]:
# Ahora podemos hacer una gráfica para comprender que hicimos:
# Me interesa ver los datos reales de los precios de las casas vs los precios que predijo el modelo. 
# Para esto, podemos hacer una gráfica de dispersión (scatter plot) con los datos reales en el eje x y los datos predichos en el eje y. 
# Si el modelo es perfecto, todos los puntos deberían estar alineados en la línea y = x (donde el valor predicho es igual al valor real).
plt.figure(figsize=(8, 8)) # Tamaño de la figura
plt.scatter(range(0,len(Y_test)),Y_test, color='blue', label='Precio real') # Gráfica de dispersión de los datos reales vs predichos
plt.scatter(range(0,len(Y_pred)),Y_pred, color='red', label='Precio predicho') # Gráfica de dispersión de los datos reales vs predichos
plt.legend() # Mostrar la leyenda para identificar los puntos reales y predichos
plt.xlabel('Índice de la muestra') # Etiqueta del eje x
plt.ylabel('Precio de la casa (miles de USD)') # Etiqueta del eje y
plt.title('Comparación entre precios reales y predichos') # Título de la gráfica
plt.show(); # Mostrar la gráfica 

En este gráfico se puede apreciar que los datos predicos nunca son exactamente iguales al set de datos de prueba proveniente de los datos reales. Por eso mismo es que se deben evaluar las métricas del modelo, el accuracy, R², error, y etc.

## 8. Evaluación del modelo (ver las métricas)

In [13]:
# R² (R-squared)
r2 = r2_score(Y_test, Y_pred) # Calcular el R²
print(f"R²: {r2:.4f}") # Imprimir el R²

R²: 0.9641


In [21]:
# Otra forma de calcular el R² es usando el método score del modelo, 
r2_o = model.score(X_test_scaled, Y_test) # Calcular el R² usando el método score del modelo (esto es equivalente a usar r2_score)
print(f"R²: {r2_o:.4f}") # Imprimir el R²

R²: 0.9641


Un R² elevado, significa que tenemos un buen ajuste de las predicciones del modelo con respecto a los datos reales.

In [14]:
# mean_squared_error
mse = mean_squared_error(Y_test, Y_pred) # Calcular el MSE
print(f"Mean Squared Error: {mse:.4f}") # Imprimir el MSE              

Mean Squared Error: 109.0298


Mientras más cercano al cero es mejor; pero para entender bien que significa este error; se debe sacar la raíz cuadrada de dicho error y contrastar contra el orden de magnitud de la variable objetivo (precio_usd)
En este ejemplo es de aprox 100 unidades al cuadrado; su error (no cuadrado) es de 10 unidades;
Si hablamos de miles de dólares el error se puede interpretar como que el modelo tiene un error con respecto al valo real de más o menos 10 mil dólares. 

Eso se puede apreciar en el gráfico; claro que algunas veces encontraremos errores más grandes y otros más chicos. Pero en promedio, el valor del precio de la casa calculada por el modelo, le falla al precio real por unos 10 mil USD; eso significa que el resultado es bastante bueno.

In [15]:
# mean_absolute_error
mae = mean_absolute_error(Y_test, Y_pred) # Calcular el MAE
print(f"Mean Absolute Error: {mae:.4f}") # Imprimir el MA

Mean Absolute Error: 8.4664


Es parecido al MSE, pero este no penaliza los errores grandes (cuando hay muchas diferencias entre el valor real y el predicho)
MAE de 8 quiere decir que el modelo falla en predecir el valor real más o menos en 8 unidades (8000 usd) en promedio.

In [16]:
# median_absolute_error
medae = median_absolute_error(Y_test, Y_pred) # Calcular el MedAE
print(f"Median Absolute Error: {medae:.4f}") # Imprimir el MedAE

Median Absolute Error: 7.7404


Es parecido al MAE; pero este es más usado en la realidad; porque en muchos casos es mejor usar la mediana en lugar del promedio para describir los datos. Si tienes dudas sobre la diferencia de la mediana y el promedio, busca un poquito en el google para que te convenzas.


In [17]:
# mean_absolute_percentage_error
mape = mean_absolute_percentage_error(Y_test, Y_pred) # Calcular el MAPE
print(f"Mean Absolute Percentage Error: {mape:.4f}") # Imprimir el

Mean Absolute Percentage Error: 0.0301


Indica el error promedio en porcentaje. En este caso es muy útil porque ningún valor de la variable objetivo es cercana a cero.

Últimos comentarios:
¿Por qué no uso otros indicadores como ROC o accuracy_score?

Porque ambas métricas se usan en problemas de clasificación. Acá se podrían utilizar si se binariza la variable objetivo (0 y 1); entonces el problema se convertiría en un problema de categorización.
